In [1]:
from imblearn.over_sampling import SMOTE
print("SMOTE imported successfully!")

SMOTE imported successfully!


In [2]:
# Cell 1: Imports & Setup
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import os

os.makedirs("../reports", exist_ok=True)
print("Task 1b: Imports complete")

Task 1b: Imports complete


In [3]:
# Cell 2: Load & Quick Cleaning (Fraud_Data.csv)
fraud_df = pd.read_csv("../data/raw/Fraud_Data.csv")
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['ip_address'] = fraud_df['ip_address'].astype('int64')
print("Loaded & cleaned Fraud_Data shape:", fraud_df.shape)

Loaded & cleaned Fraud_Data shape: (151112, 11)


In [4]:
# Cell 3: Feature Engineering (rubric requirement)
fraud_df['time_since_signup_hours'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek
fraud_df['velocity'] = fraud_df['purchase_value'] / (fraud_df['time_since_signup_hours'] + 1)  # avoid div by 0

print("New engineered features:", ['time_since_signup_hours', 'hour_of_day', 'day_of_week', 'velocity'])

New engineered features: ['time_since_signup_hours', 'hour_of_day', 'day_of_week', 'velocity']


In [5]:
# Cell 4: Define Features & Stratified Split
num_features = ['purchase_value', 'age', 'time_since_signup_hours', 'velocity']
cat_features = ['source', 'browser', 'sex']

X = fraud_df[num_features + cat_features]
y = fraud_df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (120889, 7) Test: (30223, 7)


In [7]:
# Cell 5: Data Transformation Pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)
print("Transformation complete. Train shape:", X_train_transformed.shape)

Transformation complete. Train shape: (120889, 11)


In [10]:
# Cell 6: Class Imbalance Handling – SMOTE (only on train!)
print("\nClass Distribution BEFORE SMOTE (Fraud_Data train):")
before = pd.Series(y_train).value_counts(normalize=True).to_frame('Proportion Before SMOTE')
print(before)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_transformed, y_train)

print("\nClass Distribution AFTER SMOTE:")
after = pd.Series(y_train_smote).value_counts(normalize=True).to_frame('Proportion After SMOTE')
print(after)

# Save table for interim report
imbalance_table = pd.concat([before, after], axis=1)
imbalance_table.to_markdown('../reports/task1b_imbalance_table.md')
print("Imbalance table saved to reports/task1b_imbalance_table.md")

# Interpretation:
# - Before SMOTE: ~90.61% legitimate, ~9.39% fraud → severe imbalance
# - After SMOTE: balanced 50/50 → ideal for training (no data leakage to test)


Class Distribution BEFORE SMOTE (Fraud_Data train):
       Proportion Before SMOTE
class                         
0                     0.906352
1                     0.093648

Class Distribution AFTER SMOTE:
       Proportion After SMOTE
class                        
0                         0.5
1                         0.5
Imbalance table saved to reports/task1b_imbalance_table.md
